#Ortak Preprocessing ve Modelleme Hazirligi

Bu notebook'ta train_transaction ve train_identity dosyalarini okuyup TransactionID uzerinden left join yapacagiz.
Ardindan veriyi train / validation / test olarak bolecegiz ve ayni preprocessing yapisini hem kucuk Random Forest hem de normal Random Forest modelleri icin kullanacagiz.


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

RANDOM_STATE = 42

DATA_DIR = Path("../data/raw")
TRAIN_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = DATA_DIR / "train_identity.csv"
OUTPUT_METRICS_DIR = Path("../outputs/metrics")
PROCESSED_DATA_DIR = Path("../data/processed")

TARGET_COL = "isFraud"



In [15]:
transaction = pd.read_csv(TRAIN_TRANSACTION_PATH)
identity = pd.read_csv(TRAIN_IDENTITY_PATH)

df = transaction.merge(
    identity,
    on="TransactionID",
    how="left"
)

print("Transaction dosyasi basariyla okundu.")
print("Identity dosyasi basariyla okundu.")
print("Transaction shape:", transaction.shape)
print("Identity shape:", identity.shape)
print("Merged shape:", df.shape)
print("Merge satir sayisi dogru mu", df.shape[0] == transaction.shape[0])

df.head()


Transaction dosyasi basariyla okundu.
Identity dosyasi basariyla okundu.
Transaction shape: (590540, 394)
Identity shape: (144233, 41)
Merged shape: (590540, 434)
Merge satir sayisi dogru mu True


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [16]:
print("Hedef sutun:", TARGET_COL)
print("Hedef sutun veri icinde var mi", TARGET_COL in df.columns)

print("\nTarget sayilari:")
target_counts = df[TARGET_COL].value_counts()
print(target_counts)

print("\nTarget oranlari:")
target_ratios = df[TARGET_COL].value_counts(normalize=True)
print(target_ratios)

print("\nDuplicate satir:", df.duplicated().sum())

print("\nDtype dagilimi:")
print(df.dtypes.value_counts())

missing_ratio = df.isna().mean().sort_values(ascending=False)
missing_summary = (
    missing_ratio.head(30)
    .rename("missing_ratio")
    .reset_index()
    .rename(columns={"index": "column"})
)

print("\nEn yuksek missing ratio ilk 30 kolon:")
print(missing_summary)



Hedef sutun: isFraud
Hedef sutun veri icinde var mi True

Target sayilari:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Target oranlari:
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

Duplicate satir: 0

Dtype dagilimi:
float64    399
str         31
int64        4
Name: count, dtype: int64

En yuksek missing ratio ilk 30 kolon:
   column  missing_ratio
0   id_24       0.991962
1   id_25       0.991310
2   id_07       0.991271
3   id_08       0.991271
4   id_21       0.991264
5   id_26       0.991257
6   id_27       0.991247
7   id_23       0.991247
8   id_22       0.991247
9   dist2       0.936284
10     D7       0.934099
11  id_18       0.923607
12    D13       0.895093
13    D14       0.894695
14    D12       0.890410
15  id_04       0.887689
16  id_03       0.887689
17     D6       0.876068
18  id_33       0.875895
19  id_09       0.873123
20     D8       0.873123
21  id_10       0.873123
22     D9       0.873123
23  id_30       0.868654
24  id_32 

In [17]:
X = df.drop(columns=[TARGET_COL, "TransactionID"], errors="ignore")
y = df[TARGET_COL]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_valid.shape, y_valid.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nFraud oranları:")
print("Train:", y_train.mean())
print("Validation:", y_valid.mean())
print("Test:", y_test.mean())


Train shape: (354324, 432) (354324,)
Validation shape: (118108, 432) (118108,)
Test shape: (118108, 432) (118108,)

Fraud oranları:
Train: 0.03499057359930459
Validation: 0.0349933958749619
Test: 0.03498492904798998


In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "str", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Sayisal sutun sayisi:", len(numeric_features))
print("Kategorik sutun sayisi:", len(categorical_features))
print("Toplam ozellik sayisi:", len(numeric_features) + len(categorical_features))

categorical_features


Sayisal sutun sayisi: 401
Kategorik sutun sayisi: 31
Toplam ozellik sayisi: 432


['ProductCD',
 'card4',
 'card6',
 'P_emaildomain',
 'R_emaildomain',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'id_12',
 'id_15',
 'id_16',
 'id_23',
 'id_27',
 'id_28',
 'id_29',
 'id_30',
 'id_31',
 'id_33',
 'id_34',
 'id_35',
 'id_36',
 'id_37',
 'id_38',
 'DeviceType',
 'DeviceInfo']

In [19]:
OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

phase1_report = pd.DataFrame([
    {"kontrol": "transaction_shape", "deger": str(transaction.shape)},
    {"kontrol": "identity_shape", "deger": str(identity.shape)},
    {"kontrol": "merged_shape", "deger": str(df.shape)},
    {"kontrol": "merge_satir_sayisi_dogru_mu", "deger": str(df.shape[0] == transaction.shape[0])},
    {"kontrol": "target_0_sayisi", "deger": int(target_counts.get(0, 0))},
    {"kontrol": "target_1_sayisi", "deger": int(target_counts.get(1, 0))},
    {"kontrol": "fraud_orani", "deger": float(target_ratios.get(1, 0.0))},
    {"kontrol": "duplicate_satir", "deger": int(df.duplicated().sum())},
    {"kontrol": "train_shape", "deger": str(X_train.shape)},
    {"kontrol": "validation_shape", "deger": str(X_valid.shape)},
    {"kontrol": "test_shape", "deger": str(X_test.shape)},
    {"kontrol": "train_fraud_orani", "deger": float(y_train.mean())},
    {"kontrol": "validation_fraud_orani", "deger": float(y_valid.mean())},
    {"kontrol": "test_fraud_orani", "deger": float(y_test.mean())},
    {"kontrol": "numeric_feature_count", "deger": len(numeric_features)},
    {"kontrol": "categorical_feature_count", "deger": len(categorical_features)},
])

feature_columns = pd.DataFrame({
    "column": X_train.columns,
    "dtype": X_train.dtypes.astype(str).values,
    "feature_type": ["categorical" if col in categorical_features else "numeric" for col in X_train.columns]
})

phase1_report_path = OUTPUT_METRICS_DIR / "phase1_data_checks.csv"
missing_report_path = OUTPUT_METRICS_DIR / "phase1_missing_top30.csv"
feature_columns_path = PROCESSED_DATA_DIR / "phase1_feature_columns.csv"

phase1_report.to_csv(phase1_report_path, index=False)
missing_summary.to_csv(missing_report_path, index=False)
feature_columns.to_csv(feature_columns_path, index=False)

print("Asama 1 kontrol raporu kaydedildi:", phase1_report_path)
print("Missing value raporu kaydedildi:", missing_report_path)
print("Feature kolon listesi kaydedildi:", feature_columns_path)


Asama 1 kontrol raporu kaydedildi: ..\outputs\metrics\phase1_data_checks.csv
Missing value raporu kaydedildi: ..\outputs\metrics\phase1_missing_top30.csv
Feature kolon listesi kaydedildi: ..\data\processed\phase1_feature_columns.csv


In [20]:
def evaluate_model(model, X_valid, y_valid, model_name):
    y_pred = model.predict(X_valid)
    y_proba = model.predict_proba(X_valid)[:, 1]
    
    roc_auc = roc_auc_score(y_valid, y_proba)
    avg_precision = average_precision_score(y_valid, y_proba)
    
    print(f"Model: {model_name}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"Average Precision: {avg_precision:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_valid, y_pred, digits=4))
    
    return {
        "model": model_name,
        "roc_auc": roc_auc,
        "average_precision": avg_precision
    }


In [21]:
small_rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=30,
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

small_rf_model.fit(X_train, y_train)

small_rf_results = evaluate_model(
    small_rf_model,
    X_valid,
    y_valid,
    "Küçük Random Forest"
)

small_rf_results


Model: Küçük Random Forest
ROC-AUC: 0.8639
Average Precision: 0.4374

Classification Report:
              precision    recall  f1-score   support

           0     0.9885    0.8414    0.9090    113975
           1     0.1430    0.7297    0.2392      4133

    accuracy                         0.8375    118108
   macro avg     0.5657    0.7856    0.5741    118108
weighted avg     0.9589    0.8375    0.8856    118108



{'model': 'Küçük Random Forest',
 'roc_auc': 0.8638746255973314,
 'average_precision': 0.4373927389955751}

In [22]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_results = evaluate_model(
    rf_model,
    X_valid,
    y_valid,
    "Random Forest"
)

rf_results


Model: Random Forest
ROC-AUC: 0.9304
Average Precision: 0.6229

Classification Report:
              precision    recall  f1-score   support

           0     0.9867    0.9747    0.9806    113975
           1     0.4771    0.6371    0.5456      4133

    accuracy                         0.9629    118108
   macro avg     0.7319    0.8059    0.7631    118108
weighted avg     0.9688    0.9629    0.9654    118108



{'model': 'Random Forest',
 'roc_auc': 0.9303790339918907,
 'average_precision': 0.6228513993489551}